# RFM analüüs UrbanStyle.ltd jaoks

**Nädal 7: Python Pandas**  
**Meeskonna töö: Data Loading, Data Cleaning, RFM Analysis, Visualization**  
**Kuupäev:** 2026-05-13

Marko Saar soovib customer-level analüüsi, et eristada VIP-kliendid, riskikliendid ja muud olulised kliendisegmendid. Selle notebooki eesmärk on laadida UrbanStyle.ltd müügi- ja kliendiandmed, puhastada need, arvutada RFM mõõdikud ning anda Markole konkreetsed soovitused.

## Roll A: Data Loading - andmete laadimine ja liitmine

Laeme müügi- ja kliendiandmed Supabase'ist ning liidame need `customer_id` põhjal üheks DataFrame'iks.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
from supabase import create_client

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

dotenv_candidates = [
    Path.cwd() / '.env',
    Path.cwd() / 'week7-python' / '.env',
    Path.cwd().parent / '.env',
    Path.cwd().parent / 'week7-python' / '.env',
    *[parent / '.env' for parent in Path.cwd().parents],
]
dotenv_path = next((path for path in dotenv_candidates if path.exists()), None)
load_dotenv(dotenv_path=dotenv_path)

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY') or os.getenv('SUPABASE_ANON_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError('Puudub SUPABASE_URL või SUPABASE_KEY .env failis')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)


def fetch_table(table_name, page_size=1000):
    rows = []
    start = 0

    while True:
        end = start + page_size - 1
        response = supabase.table(table_name).select('*').range(start, end).execute()
        batch = response.data or []
        rows.extend(batch)

        if len(batch) < page_size:
            break

        start += page_size

    return pd.DataFrame(rows)


df_sales = fetch_table('sales')
df_customers = fetch_table('customers')

print('\nSales shape:', df_sales.shape)
display(df_sales.head())
print('\nCustomers shape:', df_customers.shape)
display(df_customers.head())

df = pd.merge(df_sales, df_customers, on='customer_id', how='left')

print('\nLiidetud DataFrame shape:', df.shape)
print('\nLiidetud DataFrame dtypes:')
print(df.dtypes)
display(df.head())

required_columns = ['customer_id', 'sale_date', 'total_price', 'email']
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Puuduvad kohustuslikud veerud: {missing_columns}')

print('\nKontroll läbitud: customer_id, sale_date, total_price ja email on olemas.')


Sales shape: (10118, 12)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,15235,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,15236,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,15237,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,15238,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,15239,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart



Customers shape: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992



Liidetud DataFrame shape: (10118, 20)

Liidetud DataFrame dtypes:
id                     int64
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,15235,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,15236,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,15237,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,15238,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,15239,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0



Kontroll läbitud: customer_id, sale_date, total_price ja email on olemas.


## Roll B: Data Cleaning - puhastamine ja valideerimine

Eemaldame duplikaadid, käsitleme kriitilised NULL väärtused, parsime kuupäevad ning jätame alles ainult positiivse `total_price` väärtusega read.

In [8]:
print('Esialgne shape:', df.shape)

initial_rows = len(df)
duplicates_before = df.duplicated().sum()
print('Duplikaadid enne eemaldamist:', duplicates_before)

df_clean = df.drop_duplicates().copy()
duplicates_after = df_clean.duplicated().sum()
print('Duplikaadid pärast eemaldamist:', duplicates_after)

print('\nNULL-id enne puhastamist:')
nulls_before = df_clean.isnull().sum()
print(nulls_before[nulls_before > 0].sort_values(ascending=False))

critical_columns = ['customer_id', 'sale_date', 'total_price']
critical_nulls_before = df_clean[critical_columns].isnull().sum()
df_clean = df_clean.dropna(subset=critical_columns).copy()

df_clean['sale_date'] = pd.to_datetime(df_clean['sale_date'], errors='coerce')
invalid_dates = df_clean['sale_date'].isna().sum()
df_clean = df_clean.dropna(subset=['sale_date']).copy()

df_clean['total_price'] = pd.to_numeric(df_clean['total_price'], errors='coerce')
invalid_prices = df_clean['total_price'].isna().sum()
df_clean = df_clean.dropna(subset=['total_price']).copy()

non_positive_prices = (df_clean['total_price'] <= 0).sum()
df_clean = df_clean[df_clean['total_price'] > 0].copy()

final_rows = len(df_clean)
removed_rows = initial_rows - final_rows

print('\nPuhastusraport')
print('-' * 40)
print(f'Algne ridade arv: {initial_rows}')
print(f'Eemaldatud duplikaate: {duplicates_before}')
print('Kriitilised NULL-id enne eemaldamist:')
print(critical_nulls_before)
print(f'Vigaseid kuupäevi pärast parsimist: {invalid_dates}')
print(f'Mittearvulisi total_price väärtusi: {invalid_prices}')
print(f'Mittepositiivseid total_price väärtusi: {non_positive_prices}')
print(f'Eemaldatud ridu kokku: {removed_rows}')
print(f'Lõplik shape: {df_clean.shape}')
print(f'Unikaalseid kliente: {df_clean["customer_id"].nunique()}')
print(f'Kuupäevavahemik: {df_clean["sale_date"].min().date()} kuni {df_clean["sale_date"].max().date()}')

print('\nKontroll: kriitiliste veergude NULL-id pärast puhastamist')
print(df_clean[critical_columns].isnull().sum())
print('\nsale_date tüüp:', df_clean['sale_date'].dtype)
print('Min total_price:', df_clean['total_price'].min())

df = df_clean.reset_index(drop=True)
display(df.head())

Esialgne shape: (10118, 20)
Duplikaadid enne eemaldamist: 0
Duplikaadid pärast eemaldamist: 0

NULL-id enne puhastamist:
loyalty_tier         4660
store_location       3462
email                1944
customer_id           988
last_name             988
first_name            988
phone                 988
city                  988
registration_date     988
birth_year            988
dtype: int64

Puhastusraport
----------------------------------------
Algne ridade arv: 10118
Eemaldatud duplikaate: 0
Kriitilised NULL-id enne eemaldamist:
customer_id    988
sale_date        0
total_price      0
dtype: int64
Vigaseid kuupäevi pärast parsimist: 0
Mittearvulisi total_price väärtusi: 0
Mittepositiivseid total_price väärtusi: 180
Eemaldatud ridu kokku: 1168
Lõplik shape: (8950, 20)
Unikaalseid kliente: 2540
Kuupäevavahemik: 2023-01-01 kuni 2026-04-14

Kontroll: kriitiliste veergude NULL-id pärast puhastamist
customer_id    0
sale_date      0
total_price    0
dtype: int64

sale_date tüüp: datetime6

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,None,+37254290294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+37251501812,Tallinn,2020-09-22,None,1996.0
2,3,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+37288097990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+37283754888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+37253780596,Tartu,2021-04-09,None,1996.0


## Roll C: RFM Analysis - Recency, Frequency, Monetary ja segmendid

Arvutame iga kliendi kohta RFM väärtused, määrame kvintiilide alusel skoorid 1-5 ning loome kliendisegmendid.

In [9]:
today = pd.to_datetime('2025-02-28')

rfm = (
    df.groupby('customer_id')
    .agg(
        last_purchase_date=('sale_date', 'max'),
        frequency=('sale_id', 'count'),
        monetary_value=('total_price', 'sum'),
        first_name=('first_name', 'first'),
        last_name=('last_name', 'first'),
        email=('email', 'first'),
        city=('city', 'first'),
        loyalty_tier=('loyalty_tier', 'first'),
    )
    .reset_index()
)

rfm['recency_days'] = (today - rfm['last_purchase_date']).dt.days

rfm['R_score'] = pd.qcut(
    rfm['recency_days'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1],
).astype(int)
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)
rfm['M_score'] = pd.qcut(
    rfm['monetary_value'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5],
).astype(int)

rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)


def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    if score >= 10:
        return 'Loyal'
    if score >= 7:
        return 'Potential'
    if score >= 4:
        return 'At Risk'
    return 'Lost'


rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

segment_summary = (
    rfm.groupby('Segment')
    .agg(
        customers=('customer_id', 'count'),
        avg_recency_days=('recency_days', 'mean'),
        avg_frequency=('frequency', 'mean'),
        total_revenue=('monetary_value', 'sum'),
        avg_monetary_value=('monetary_value', 'mean'),
    )
    .reset_index()
)
segment_summary['customer_share_pct'] = segment_summary['customers'] / segment_summary['customers'].sum() * 100
segment_summary['revenue_share_pct'] = segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100
segment_summary = segment_summary.sort_values('total_revenue', ascending=False)

print('RFM tabel shape:', rfm.shape)
print('\nSkooride vahemikud:')
print(rfm[['R_score', 'F_score', 'M_score', 'RFM_Score']].agg(['min', 'max']))

print('\nSegmentide kokkuvõte:')
display(segment_summary.round(2))

display(rfm.sort_values('RFM_Score', ascending=False).head(10))

RFM tabel shape: (2540, 15)

Skooride vahemikud:
     R_score  F_score  M_score  RFM_Score
min        1        1        1          3
max        5        5        5         15

Segmentide kokkuvõte:


,Segment,customers,avg_recency_days,avg_frequency,total_revenue,avg_monetary_value,customer_share_pct,revenue_share_pct
4,VIP Champions,455,48.92,7.68,1146295.15,2519.33,17.91,42.82
2,Loyal,678,145.37,3.84,795523.58,1173.34,26.69,29.72
3,Potential,758,207.33,2.49,521614.40,688.15,29.84,19.49
0,At Risk,533,310.35,1.59,193876.62,363.75,20.98,7.24
1,Lost,116,516.10,1.01,19540.79,168.46,4.57,0.73


,customer_id,last_purchase_date,frequency,monetary_value,first_name,last_name,email,city,loyalty_tier,recency_days,R_score,F_score,M_score,RFM_Score,Segment
2030,4400.0,2025-02-17,7,1806.37,Mihkel,Kask,mihkel.kask@yahoo.com,Viljandi,None,11,5,5,5,15,VIP Champions
299,2349.0,2025-02-18,5,1932.51,Ene,Rebane,ene.rebane@gmail.com,Jõhvi,silver,10,5,5,5,15,VIP Champions
1882,4220.0,2025-02-20,6,1970.42,Kadri,Kukk,kadri.kukk@gmail.com,Tallinn,gold,8,5,5,5,15,VIP Champions
98,2115.0,2025-02-15,6,1635.44,Anu,Sild,anu.sild@yahoo.com,Haapsalu,silver,13,5,5,5,15,VIP Champions
1532,3810.0,2025-02-17,55,18110.41,Pille,Sepp,pille.sepp@hot.ee,Pärnu,None,11,5,5,5,15,VIP Champions
2186,4589.0,2025-02-20,7,2131.86,Lauri,Kangur,lauri.kangur@telia.ee,Viljandi,gold,8,5,5,5,15,VIP Champions
1549,3828.0,2025-02-10,5,1443.54,Pille,Põld,pille.pold@yahoo.com,Tallinn,None,18,5,5,5,15,VIP Champions
2492,4949.0,2025-02-25,65,19580.28,Marika,Sepp,marika.sepp@gmail.com,Tallinn,bronze,3,5,5,5,15,VIP Champions
311,2363.0,2025-01-16,5,1857.31,Tõnu,Tamm,tonu.tamm@hot.ee,Tallinn,gold,43,5,5,5,15,VIP Champions
577,2684.0,2025-01-02,6,3287.93,Andres,Sild,andres.sild@mail.ee,Pärnu,None,57,5,5,5,15,VIP Champions


## Roll D: Visualization - diagrammid ja leiud

Loome kolm Plotly diagrammi: segmentide jaotus, RFM hajuvusdiagramm ning top 10 VIP klienti kogukulutuse järgi.

In [10]:
segment_order = ['VIP Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']

fig_segments = px.bar(
    segment_summary.sort_values('customers', ascending=False),
    x='Segment',
    y='customers',
    text='customers',
    title='UrbanStyle kliendisegmentide jaotus',
    labels={'Segment': 'Segment', 'customers': 'Klientide arv'},
    color='Segment',
    category_orders={'Segment': segment_order},
)
fig_segments.update_layout(showlegend=False)
fig_segments.update_traces(textposition='outside')
fig_segments.show()

fig_scatter = px.scatter(
    rfm,
    x='recency_days',
    y='monetary_value',
    color='Segment',
    size='frequency',
    hover_data=['customer_id', 'email', 'frequency', 'RFM_Score'],
    title='UrbanStyle kliendisegmendid RFM analüüsi põhjal',
    labels={
        'recency_days': 'Päevi viimasest ostust',
        'monetary_value': 'Kogukulutus (EUR)',
        'frequency': 'Ostude arv',
        'Segment': 'Segment',
    },
    category_orders={'Segment': segment_order},
)
fig_scatter.show()

top_vip = rfm[rfm['Segment'] == 'VIP Champions'].nlargest(10, 'monetary_value').copy()
top_vip['customer_label'] = top_vip['customer_id'].astype(str) + ' - ' + top_vip['first_name'].fillna('') + ' ' + top_vip['last_name'].fillna('')

fig_vip = px.bar(
    top_vip.sort_values('monetary_value'),
    x='monetary_value',
    y='customer_label',
    orientation='h',
    text='monetary_value',
    title='Top 10 VIP klienti kogukulutuse järgi',
    labels={'monetary_value': 'Kogukulutus (EUR)', 'customer_label': 'Klient'},
)
fig_vip.update_traces(texttemplate='%{text:.0f} EUR', textposition='outside')
fig_vip.show()

<style>
.jp-RenderedHTMLCommon .github-only,
.rendered_html .github-only {
  display: none;
}
</style>

<div class="github-only">
<h3>Roll D visualisatsioonid GitHubi jaoks</h3>
<p><img src="rfm_segmentide_jaotus.png" alt="RFM segmentide jaotus"></p>
<p><img src="rfm_segmentide_scatter.png" alt="RFM kliendisegmendid scatter diagrammil"></p>
<p><img src="rfm_top_10_vip.png" alt="Top 10 VIP klienti kogukulutuse järgi"></p>
</div>


## Äritõlgendus Markole

In [11]:
vip_customers = int((rfm['Segment'] == 'VIP Champions').sum())
at_risk_customers = int((rfm['Segment'] == 'At Risk').sum())
lost_customers = int((rfm['Segment'] == 'Lost').sum())
total_customers = int(rfm['customer_id'].nunique())
total_revenue = rfm['monetary_value'].sum()
vip_revenue = rfm.loc[rfm['Segment'] == 'VIP Champions', 'monetary_value'].sum()
vip_revenue_share = vip_revenue / total_revenue * 100
at_risk_revenue = rfm.loc[rfm['Segment'] == 'At Risk', 'monetary_value'].sum()
at_risk_revenue_share = at_risk_revenue / total_revenue * 100

interpretation = f'''
UrbanStyle andmestikus on {total_customers} analüüsitavat klienti, kellest {vip_customers} kuuluvad VIP Champions segmenti.
VIP kliendid annavad {vip_revenue_share:.1f}% kogukäibest, seega on nad Marko jaoks kõige väärtuslikum hoidmise ja erikohtlemise sihtrühm.
At Risk segmendis on {at_risk_customers} klienti ja Lost segmendis {lost_customers} klienti; nende puhul on peamine risk ostmise vähenemine või lõppemine.
At Risk segment annab veel {at_risk_revenue_share:.1f}% käibest, mistõttu tasub nendega kiiresti teha personaliseeritud win-back kampaania.
'''

recommendations = '''
Soovitused:
1. VIP Champions: käivita early access programm, personaalsed pakkumised ja VIP sooduskoodid.
2. At Risk: saada "Me igatseme sind" tüüpi win-back e-mail koos piiratud ajaga personaalse pakkumisega.
3. Potential ja Loyal: kasuta lojaalsusprogrammi, cross-sell pakkumisi ja kordusostu soodustusi, et kasvatada nad VIP segmendiks.
'''

print(interpretation.strip())
print()
print(recommendations.strip())

UrbanStyle andmestikus on 2540 analüüsitavat klienti, kellest 455 kuuluvad VIP Champions segmenti.
VIP kliendid annavad 42.8% kogukäibest, seega on nad Marko jaoks kõige väärtuslikum hoidmise ja erikohtlemise sihtrühm.
At Risk segmendis on 533 klienti ja Lost segmendis 116 klienti; nende puhul on peamine risk ostmise vähenemine või lõppemine.
At Risk segment annab veel 7.2% käibest, mistõttu tasub nendega kiiresti teha personaliseeritud win-back kampaania.

Soovitused:
1. VIP Champions: käivita early access programm, personaalsed pakkumised ja VIP sooduskoodid.
2. At Risk: saada "Me igatseme sind" tüüpi win-back e-mail koos piiratud ajaga personaalse pakkumisega.
3. Potential ja Loyal: kasuta lojaalsusprogrammi, cross-sell pakkumisi ja kordusostu soodustusi, et kasvatada nad VIP segmendiks.


## Edasijõudnute lisa: segmentide eksport

Salvestame RFM segmentide tabeli CSV-failina, et Marko saaks selle edasi anda turundusmeeskonnale.

In [12]:
export_columns = [
    'customer_id', 'first_name', 'last_name', 'email', 'city', 'last_purchase_date',
    'recency_days', 'frequency', 'monetary_value', 'R_score', 'F_score', 'M_score',
    'RFM_Score', 'Segment'
]
export_path = Path('week7-python/team/rfm_segments.csv')
if not export_path.parent.exists():
    export_path = Path('rfm_segments.csv')

rfm[export_columns].to_csv(export_path, index=False, encoding='utf-8')
print(f'RFM segmendid salvestatud: {export_path}')

RFM segmendid salvestatud: rfm_segments.csv


## Kokkuvõte: soovitused Markole ja meeskonna refleksioon

**Mis oli suurim üllatus?**  
VIP Champions segment ei ole kõige suurem kliendigrupp, kuid selle käibeosakaal on väga suur. See näitab, et väike osa klientidest võib kanda ebaproportsionaalselt suurt osa müügitulust.

**Milline on meie soovitus Markole?**  
Kõige kiiremini tuleks tegutseda kahe segmendiga: VIP Champions ja At Risk. VIP klientidele tuleb pakkuda eksklusiivseid eeliseid, et neid hoida; At Risk klientidele tuleb teha personaliseeritud win-back kampaania enne, kui nad liiguvad Lost segmenti.

**Milliseid andmeid meil puudus?**  
Analüüsi muudaks paremaks info tagastuste, kampaaniate, kliendi veebikäitumise, e-mailide avamise ja NPS rahuloluskoori kohta. Need aitaksid paremini selgitada, miks osa kliente lõpetab ostmise.